# Fama–French and CAPM benchmarks

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            train_end_idx = test_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits


In [ ]:
from utils import format_time, MultipleTimeSeriesCV


# **Load stock returns data**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install pandas pyarrow
# import pandas as pd

# # Define the file path
# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# # Load the Parquet file into a DataFrame
# df = pd.read_parquet(file_path, engine='pyarrow')

# # Check the loaded data
# print(df.head())

# import pandas as pd

# # Reset the index to convert 'month' to a column
# df_reset = df.reset_index()

# # Ensure the 'month' column is in datetime format
# df_reset['month'] = pd.to_datetime(df_reset['month'])

# # Set the index back to 'month' and 'permno'
# df = df_reset.set_index(['month', 'permno'])

# # Sort the MultiIndex to ensure it is lexsorted
# df = df.sort_index()

# # Define the date range
# start_date = '1960-01-01'
# end_date = '2020-01-01'

# # Filter the DataFrame for the specified date range
# df = df.loc[start_date:end_date]

# # Keep only the "ret_excess" column
# df = df.loc[:, ['ret_excess']]

# # Display the filtered DataFrame
# print(df.index)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow scikit-learn joblib tqdm tables

import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())


# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['month', 'permno'])

# Sort the MultiIndex
df = df.sort_index()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Filter for dates starting from '1993'
df = df.loc[pd.IndexSlice['1960':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# Define the function to apply quantile transformation for a single date
def process_date(date, df, characteristics):
    group = df.loc[date, characteristics]
    transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
    transformed_df = transformed_df.mul(2).sub(1)
    transformed_df['month'] = date  # Add the date back to the DataFrame
    return transformed_df

# Initialize the progress bar
dates = df.index.get_level_values('month').unique()
pbar = tqdm(dates, desc="Processing Dates")

# Apply the transformation with parallel processing
normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# Concatenate all the normalized data back into a single DataFrame
normalized_df = pd.concat(normalized_data)

# Reset and set the index to match the original DataFrame's multi-index
normalized_df = normalized_df.reset_index().set_index(['month', 'permno'])

# Ensure the index order matches the original DataFrame
normalized_df = normalized_df.reindex(df.index)

# Assign the normalized values back to the original DataFrame
df.loc[:, characteristics] = normalized_df[characteristics].values

# Close the progress bar
pbar.close()

# # Scale the variables
# scaler = StandardScaler()
# df[characteristics] = scaler.fit_transform(df[characteristics])

# Clip the returns columns
df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)

# Print the first few rows of the scaled DataFrame to verify
print(df.head())

# Add 'ret_excess' to the list of columns
columns_to_select = characteristics

# Select these columns from the DataFrame
characteristics_data = df[columns_to_select]

# Define a function to fill missing values with the median for each group
def fill_missing_with_median(group):
    return group.fillna(group.median())

# Apply the function to each group (each date) and preserve the index
df_filled = characteristics_data.groupby(level='month').apply(fill_missing_with_median)

# Remove the additional 'date' level in the index introduced by groupby
df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
df.update(df_filled)

print(df.index)




# **Downloading Fama-French factors**

In [ ]:
import pandas as pd
import requests
import zipfile
from io import BytesIO

# Define URLs for the data
ff5_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip'
momentum_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip'

# Function to download and return processed dataframe
def download_and_process_dataframe(url, skiprows):
    response = requests.get(url)
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        filename = z.namelist()[0]
        with z.open(filename) as file:
            df = pd.read_csv(file, skiprows=skiprows)
    return df

# Download and print the FF5 processed data
ff5_processed_df = download_and_process_dataframe(ff5_url, skiprows=3)
print("FF5 Processed DataFrame:")
print(ff5_processed_df.head(20))

# Download and print the Momentum processed data
momentum_processed_df = download_and_process_dataframe(momentum_url, skiprows=12)
print("\nMomentum Processed DataFrame:")
print(momentum_processed_df.head(20))

import pandas as pd

# Function to clean and prepare dataframe
def clean_dataframe(df, date_column):
    # Rename the date column
    df.rename(columns={date_column: 'month'}, inplace=True)
    # Convert the date column to datetime format
    df['month'] = pd.to_datetime(df['month'], format='%Y%m', errors='coerce')
    # Remove rows with invalid dates
    df = df.dropna(subset=['month'])
    # Set the date column as the index
    df.set_index('month', inplace=True)
    # Convert all columns to numeric values
    df = df.apply(pd.to_numeric, errors='coerce')
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    return df

# Clean and process the FF5 dataframe
ff5_cleaned_df = clean_dataframe(ff5_processed_df, 'Unnamed: 0')
print("FF5 Cleaned DataFrame:")
print(ff5_cleaned_df.head(20))

# Clean and process the Momentum dataframe
momentum_cleaned_df = clean_dataframe(momentum_processed_df, 'Unnamed: 0')
print("\nMomentum Cleaned DataFrame:")
print(momentum_cleaned_df.head(20))

# Merge FF5 and Momentum dataframes
merged_df = ff5_cleaned_df.join(momentum_cleaned_df['Mom'])
print("\nMerged DataFrame:")
print(merged_df.head(20))

# Check the frequency of the date index in the merged dataframe
print("Frequency of the date index:", pd.infer_freq(merged_df.index))

# Print the first few rows of the merged dataframe
print("\nFirst few rows of the merged dataframe:")
print(merged_df.head())

# Print the last few rows of the merged dataframe
print("\nLast few rows of the merged dataframe:")
print(merged_df.tail())


# Merge the factors dataframe with the returns dataframe
df_merged = df.reset_index().merge(merged_df, on='month').set_index(['month', 'permno'])

# Strip any extra whitespace from column names
df_merged.columns = df_merged.columns.str.strip()

# Rename the "month" level of the multi-index
df_merged = df_merged.rename_axis(index={'month': 'date'})

# Display the merged dataframe
print(df_merged.head())

# Install the linearmodels package if not already installed
# !pip install linearmodels  # Uncomment this line if running in a new environment



# **Creating Forward returns for predictive $R^2$**

In [ ]:
import pandas as pd

# Assuming df_merged is your DataFrame

# Step 1: Sort the DataFrame by permno and date
df_merged = df_merged.sort_values(by=['permno', 'date'])

# Step 2: Group by permno and calculate forward returns
df_merged['forward_ret_excess'] = df_merged.groupby('permno')['ret_excess'].shift(-1)

# Step 3: Drop rows with NaN forward returns (typically the last month for each permno)
df_merged = df_merged.dropna(subset=['forward_ret_excess'])

# Display the DataFrame with forward returns
print(df_merged.head())
print(df_merged.tail())


# **Predictions of FF models. COMMENT: for predictions with forward returns, set the file name in the hadf5 path to fwd_predictions.h5 and for contemporary returns to predictions.h5**

In [ ]:
from pathlib import Path
import pandas as pd

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIM_PRED')
hdf5_path = results_path / 'asset_pricing.h5'

# Check if results_path exists, create it if it doesn't
if not results_path.exists():
    results_path.mkdir(parents=True)

# Check if hdf5_path exists, create the HDF5 file if it doesn't
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        pass  # The file is created and then immediately closed


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path

# Function to preprocess the train and test data
def get_train_test_data(df, train_idx, test_idx):
    train_set = df.iloc[train_idx]
    test_set = df.iloc[test_idx]

    y_train = train_set['forward_ret_excess']
    x_train = train_set.drop(columns=['forward_ret_excess'])

    y_test = test_set['forward_ret_excess']
    x_test = test_set.drop(columns=['forward_ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(df_merged))

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
hdf5_path = results_path / 'fwd_predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors sequences to use
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'RMW','Mom']
]
# Loop over each factor sequence and calculate average R^2 for all folds
for factor_seq in factor_sequences:
    n_factors = len(factor_seq)
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(df_merged[factor_seq + ['forward_ret_excess']], train_idx, test_idx)

        # Initialize the Linear Regression model
        model = LinearRegression()

        # Fit the model on the training data
        model.fit(x_train, y_train)

        # Generate out-of-sample predictions
        y_pred = model.predict(x_test)

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'y_pred': y_pred,
            'y_true': y_test
        }).reset_index(drop=True)

        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/n_factors_{n_factors}_fold_{fold}', prediction_df, format='table')
            print(f"Saved: predictions/n_factors_{n_factors}_fold_{fold}")

        print(f"Fold {fold+1}, Factors {' '.join(factor_seq)} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for factors {' '.join(factor_seq)} across all folds: {average_r2}")


In [ ]:
# Display the DataFrame with forward returns
print(df_merged.head())
print(df_merged.tail())


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path

# Function to preprocess the train and test data
def get_train_test_data(df, train_idx, test_idx):
    train_set = df.iloc[train_idx]
    test_set = df.iloc[test_idx]

    y_train = train_set['ret_excess']
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess']
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(df_merged))

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors sequences to use
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'Mom']
]

# Loop over each factor sequence and calculate average R^2 for all folds
for factor_seq in factor_sequences:
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(df_merged[factor_seq + ['ret_excess']], train_idx, test_idx)

        # Initialize the Linear Regression model
        model = LinearRegression()

        # Fit the model on the training data
        model.fit(x_train, y_train)

        # Generate out-of-sample predictions
        y_pred = model.predict(x_test)

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'y_pred': y_pred,
            'y_true': y_test
        }).reset_index(drop=True)

        n_factors = len(factor_seq)
        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/n_factors_{n_factors}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {' '.join(factor_seq)} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for factors {' '.join(factor_seq)} across all folds: {average_r2}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path

# Function to preprocess the train and test data
def get_train_test_data(df, train_idx, test_idx):
    train_set = df.iloc[train_idx]
    test_set = df.iloc[test_idx]

    y_train = train_set['ret_excess']
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess']
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=10, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(df_merged))

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors sequences to use
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'Mom']
]

# Loop over each factor sequence and calculate average R^2 for all folds
for factor_seq in factor_sequences:
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(df_merged[factor_seq + ['ret_excess']], train_idx, test_idx)

        # Initialize the Linear Regression model
        model = LinearRegression()

        # Fit the model on the training data
        model.fit(x_train, y_train)

        # Generate out-of-sample predictions
        y_pred = model.predict(x_test)

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'predicted returns': y_pred,
            'actual returns': y_test
        }).reset_index(drop=True)

        factor_seq_name = "_".join(factor_seq).replace("-", "_")  # Ensure valid object names
        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/factors_{factor_seq_name}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {' '.join(factor_seq)} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for factors {' '.join(factor_seq)} across all folds: {average_r2}")


In [ ]:
# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_FF')
hdf5_path = results_path / 'predictions.h5'

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path

# Function to preprocess the train and test data
def get_train_test_data(df, train_idx, test_idx):
    train_set = df.iloc[train_idx]
    test_set = df.iloc[test_idx]

    y_train = train_set['ret_excess']
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess']
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=10, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(df_merged))

# Path to save the HDF5 file
results_path = Path('./results_IPCA')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors sequences to use
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'Mom']
]

# Loop over each factor sequence and calculate average R^2 for all folds
for factor_seq in factor_sequences:
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(df_merged[factor_seq + ['ret_excess']], train_idx, test_idx)

        # Initialize the Linear Regression model
        model = LinearRegression()

        # Fit the model on the training data
        model.fit(x_train, y_train)

        # Generate out-of-sample predictions
        y_pred = model.predict(x_test)

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'predicted returns': y_pred,
            'actual returns': y_test
        }).reset_index(drop=True)

        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/factors_{"_".join(factor_seq)}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {' '.join(factor_seq)} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for factors {' '.join(factor_seq)} across all folds: {average_r2}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path

# Function to preprocess the train and test data
def get_train_test_data(df, train_idx, test_idx):
    train_set = df.iloc[train_idx]
    test_set = df.iloc[test_idx]

    y_train = train_set['ret_excess']
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess']
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=10, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(df_merged))

# Path to save the HDF5 file
results_path = Path('./results_IPCA')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors sequences to use
factor_sequences = [
    ['Mkt-RF'],
    ['Mkt-RF', 'SMB'],
    ['Mkt-RF', 'SMB', 'HML'],
    ['Mkt-RF', 'SMB', 'HML', 'Mom'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA'],
    ['Mkt-RF', 'SMB', 'HML', 'CMA', 'Mom']
]

# Loop over each factor sequence and calculate average R^2 for all folds
for factor_seq in factor_sequences:
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(df_merged[factor_seq + ['ret_excess']], train_idx, test_idx)

        # Initialize the Linear Regression model
        model = LinearRegression()

        # Fit the model on the training data
        model.fit(x_train, y_train)

        # Generate out-of-sample predictions
        y_pred = model.predict(x_test)

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('month'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'predicted returns': y_pred,
            'actual returns': y_test
        }).reset_index(drop=True)

        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/factors_{"_".join(factor_seq)}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {' '.join(factor_seq)} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for factors {' '.join(factor_seq)} across all folds: {average_r2}")


In [ ]:
# Prepare for regression
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

# Define the dependent variable
y = df_merged['ret_excess']

# Define the independent variables
X = df_merged[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']]
X = sm.add_constant(X)

# Fit the fixed effects model
model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True)
results = model.fit()

# Print the results summary
print(results.summary)


In [ ]:
import os

# List files in the current directory
print(os.listdir('.'))


In [ ]:
import pandas as pd

# Load the ff5.csv file into a dataframe
ff5_df = pd.read_csv('ff5.csv')
print(ff5_df.head())

# Load the UMD.csv file into a dataframe
umd_df = pd.read_csv('UMD.csv')
print(umd_df.head())

print(ff5_df)
print(umd_df)

In [ ]:
import pandas as pd

# Load the ff5.csv file into a dataframe
ff5_df = pd.read_csv('ff5.csv')

# Rename the "Unnamed: 0" column to "Date"
ff5_df.rename(columns={'Unnamed: 0': 'month'}, inplace=True)

# Convert the "Date" column to datetime format
ff5_df['month'] = pd.to_datetime(ff5_df['month'], format='%Y%m')

# Set the "Date" column as the index
ff5_df.set_index('month', inplace=True)

print(ff5_df.head())
print(ff5_df.index)

# Load the UMD.csv file into a dataframe
umd_df = pd.read_csv('UMD.csv')

# Rename the "Unnamed: 0" column to "Date"
umd_df.rename(columns={'Unnamed: 0': 'month'}, inplace=True)

# Convert the "Date" column to datetime format
umd_df['month'] = pd.to_datetime(umd_df['month'], format='%Y%m')

# Set the "Date" column as the index
umd_df.set_index('month', inplace=True)

print(umd_df.head())

print(umd_df.index)


In [ ]:
print(umd_df)

In [ ]:
!pip install linearmodels


In [ ]:
import pandas as pd

# Example dataframes
# Assuming df, ff5_df, and umd_df are already defined as provided

# Ensure month columns are datetime type
df.index = df.index.set_levels([pd.to_datetime(df.index.levels[0]), df.index.levels[1]])
ff5_df.index = pd.to_datetime(ff5_df.index)
umd_df.index = pd.to_datetime(umd_df.index)

# Merge the Fama-French five-factor dataframe with the momentum factor dataframe
ff_factors = ff5_df.join(umd_df, how='inner')

# Merge the factors dataframe with the returns dataframe
df_merged = df.reset_index().merge(ff_factors, on='month').set_index(['month', 'permno'])

# Strip any extra whitespace from column names
df_merged.columns = df_merged.columns.str.strip()

# Display the merged dataframe
print(df_merged.head())

# Install the linearmodels package if not already installed
# !pip install linearmodels  # Uncomment this line if running in a new environment

# Prepare for regression
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

# Define the dependent variable
y = df_merged['ret_excess']

# Define the independent variables
X = df_merged[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']]
X = sm.add_constant(X)

# Fit the fixed effects model
model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True)
results = model.fit()

# Print the results summary
print(results.summary)
